# SpatialNet Training Pipeline

This notebook covers the training of **SpatialNet**, which processes Corsi-block task metrics to assess executive planning and working memory.

## Modality & Scope
- Input: Corsi task statistics `[span_len, planning_latency, error_count, total_trials]`
- Output: Multi-class risk categorization (Typical, Mild, Moderate, Severe)
- Base Network: **Dense MLP with native Normalization layer**

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas numpy scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

## Step 2: Dataset Generation & Calibration
SpatialNet relies on age-stratified Corsi normative block-span datasets.

In [ ]:
np.random.seed(42)
features = []
labels = []
for _ in range(1000):
    lbl = np.random.choice([0, 1, 2, 3], p=[0.70, 0.12, 0.10, 0.08])
    labels.append(lbl)
    seq_len = np.random.choice([4, 5, 6, 7, 8])
    if lbl == 0:
        span = seq_len - np.random.choice([0, 1])
        planning = np.random.uniform(800, 2000)
        errors = np.random.choice([0, 1])
    elif lbl == 1:
        span = max(2, seq_len - np.random.choice([3, 4]))
        planning = np.random.uniform(1000, 2500)
        errors = np.random.choice([2, 3, 4])
    elif lbl == 2:
        span = seq_len - np.random.choice([0, 1, 2])
        planning = np.random.uniform(2500, 6000)
        errors = np.random.choice([1, 2])
    else:
        span = max(2, seq_len - np.random.choice([3, 4]))
        planning = np.random.uniform(2500, 7000)
        errors = np.random.choice([3, 4, 5])
    features.append([float(span), float(planning), float(errors), float(seq_len)])
    
X_raw = np.array(features, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

## Step 3: Model Architecture & Training

In [ ]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_raw, y, test_size=0.2, random_state=42)
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train_raw)

model = tf.keras.Sequential([
    layers.Input(shape=(4,)),
    normalizer,
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_raw, y_train, validation_data=(X_val_raw, y_val), epochs=15, batch_size=32, verbose=1)

## Step 4: Export to TFLite (Float16 Quantized)

In [ ]:
os.makedirs('output', exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
output_path = 'output/seren_spatialnet.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model successfully exported to: {output_path}")

## Step 5: Automated Behavioral Validation Check

In [ ]:
print("\n--- Running Behavioral Validation Check ---")
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
test_inputs = [
    np.array([[6.0, 1000.0, 0.0, 6.0]], dtype=np.float32),
    np.array([[2.0, 1200.0, 4.0, 6.0]], dtype=np.float32),
    np.array([[6.0, 4500.0, 1.0, 6.0]], dtype=np.float32)
]
outputs = []
for inp in test_inputs:
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    outputs.append(interpreter.get_tensor(output_details[0]['index']).flatten())

max_std = np.max(np.std(outputs, axis=0))
print(f"SpatialNet Max Std: {max_std:.4f}")
assert max_std > 0.01, "FAILED: SpatialNet output has no variance!"
print("PASSED: SpatialNet validation check successful!")